차원 축소(Dimensionality Reduction)는 비지도 학습(Unsupervised Learning)의 한 분류인 변환(Transformation) 계열의 알고리즘이다.

수백 개의 피처(feature)를 가진 데이터셋을 소수의 차원으로 근사(approximate) 하여 표현하며, 주요 활용 목적은 데이터 탐색(EDA), 시각화, 군집 패턴 파악이다.

주요 차원 축소 알고리즘

PCA(Principal Component Analysis): 분산을 최대화하는 방향으로 회전하며 빠르고 범용적
MDS(Multi-Dimensional Scaling): 원본 공간의 거리 관계를 저차원에서 보존
t-SNE(t-distributed Stochastic Neighbor Embedding): 이웃 포인트 간 거리 보존에 특화되어 있으며 지역 구조가 뚜렷한 데이터에 강점

In [ ]:
%matplotlib inline

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.datasets import load_breast_cancer

cancer = load_breast_cancer()
(X_canc, y_canc) = load_breast_cancer(return_X_y=True)

In [ ]:
def plot_labelled_scatter(X, y, class_labels, s):

    num_labels = len(class_labels)

    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1

    marker_array = ['o', '^', '*']
    color_array = ['#FFFF00', '#00AAFF', '#000000', '#FF00AA']
    cmap_bold = ListedColormap(color_array)
    bnorm = BoundaryNorm(np.arange(0, num_labels + 1, 1), ncolors=num_labels)
    plt.figure(figsize=s)

    plt.scatter(X[:, 0], X[:, 1], s=80, c=y, cmap=cmap_bold, norm=bnorm,
                alpha=0.4, edgecolor='black', lw=1)

    sp = plt.gca().spines
    sp['top'].set_visible(False)
    sp['right'].set_visible(False)

    plt.grid(which='both', color='lightslategrey', alpha=0.3)

    plt.xlim(x_min, x_max)
    plt.ylim(y_min, y_max)

    h = []
    for c in range(0, num_labels):
        h.append(mpatches.Patch(color=color_array[c], label=class_labels[c]))
    plt.legend(handles=h, fontsize=15, frameon=False)

PCA는 데이터의 분산(Variance) 을 최대한 보존하는 방향으로 좌표계를 회전시키는 선형 변환 기법

핵심 원리
1. 원본 데이터 포인트들의 분포를 바라볼 때, 가장 분산이 큰 방향 → 제1 주성분 PC1
2. PC1에 직교하며, 그 다음으로 분산이 큰 방향 → 제2 주성분 PC2
3. 이 과정을 원하는 차원 수만큼 반복

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

canc_norm = StandardScaler().fit(X_canc).transform(X_canc)

pca = PCA(n_components=2).fit(canc_norm)

canc_pca = pca.transform(canc_norm)

print('Number of Features in Breast Cancer DataSet Before PCA : {}\n\n'
      'Number of Features in Breast Cancer DataSet After PCA  : {}'
      .format(X_canc.shape[1], canc_pca.shape[1]))

In [ ]:
plot_labelled_scatter(canc_pca, y_canc, ['malignant', 'benign'], (15, 9))

plt.xlabel('First principal component', fontsize=15)
plt.ylabel('Second principal component', fontsize=15)
plt.title('Breast Cancer Dataset PCA (n_components = 2)', fontsize=17);

결론
- 30개의 피처가 2개의 주성분으로 축소되었다
- 시각화 결과가 malignant과 benign 두 클래스로 명확히 분리되었다

In [ ]:
# pca.components_: 각 주성분과 원본 피처 간의 상관 계수 행렬
fig = plt.figure(figsize=(20, 9))
plt.imshow(pca.components_, interpolation='none', cmap='viridis')

feature_names = list(cancer.feature_names)

plt.gca().set_xticks(np.arange(len(feature_names)))
plt.gca().set_yticks(np.arange(2))
plt.gca().set_xticklabels(feature_names, rotation=90, fontsize=14)
plt.gca().set_yticklabels(['First PC', 'Second PC'], fontsize=14)

plt.colorbar(orientation='horizontal',
             ticks=[pca.components_.min(), 0, pca.components_.max()],
             pad=0.5);

-PC1(첫 번째 주성분): 30개 피처 모두 양수 방향이므로 모든 피처가 함께 증가/감소하는 경향

-PC2 (두 번째 주성분): 일부 피처는 양수, 나머지는 음수 방향이므로 mean texture/worst texture, mean radius/worst radius 쌍이 같은 방향으로 함께 변화한다

MDS는 다양체 학습(Manifold Learning) 알고리즘의 일종으로, 고차원 데이터 내에 존재하는 manifold를 발견하는 데 특화되어 있다.

+) Manifold: 고차원 공간 안에 존재하는 저차원 곡면 구조  

원본 고차원 공간에서 포인트 간 거리를 최대한 보존하면서 저차원(주로 2D)으로 투영하며 결과적으로 데이터의 군집 패턴을 저차원에서도 확인 가능

In [ ]:
from sklearn.manifold import MDS

mds = MDS(n_components=2, random_state=2)

canc_mds = mds.fit_transform(canc_norm)

print('Number of Features in Breast Cancer DataSet Before MDS : {}\n\n'
      'Number of Features in Breast Cancer DataSet After MDS  : {}'
      .format(X_canc.shape[1], canc_mds.shape[1]))

plot_labelled_scatter(canc_mds, y_canc, ['malignant', 'benign'], (15, 9))

plt.xlabel('First MDS dimension', fontsize=15)
plt.ylabel('Second MDS dimension', fontsize=15)
plt.title('Breast Cancer Dataset MDS (n_components = 2)', fontsize=17);

 MDS도 두 클래스를 잘 분리하여 시각화하지만 PCA 결과와 분포 패턴이 다르다

-PCA: 분산 최대화 기반 선형 회전

-MDS: 거리 보존 기반 비선형 변환

t-SNE는 데이터 시각화를 위한 강력한 다양체 학습 알고리즘

-고차원 데이터에서 이웃(neighbor) 포인트 간 거리 관계를 2D 공간에서 최대한 보존
-특히 가까운 포인트 간의 거리 보존에 더 큰 가중치 부여
-지역적 구조(local structure) 보존에 특화

In [ ]:
from sklearn.manifold import TSNE

tsne = TSNE(random_state=42)

canc_tsne = tsne.fit_transform(canc_norm)

print('Number of Features in Breast Cancer DataSet Before T-SNE : {}\n\n'
      'Number of Features in Breast Cancer DataSet After T-SNE  : {}'
      .format(X_canc.shape[1], canc_tsne.shape[1]))

plot_labelled_scatter(canc_tsne, y_canc, ['malignant', 'benign'], (15, 9))

plt.xlabel('First T-SNE Dimension', fontsize=14)
plt.ylabel('Second T-SNE Dimension', fontsize=14)
plt.title('Breast Cancer Dataset T-SNE', fontsize=17);

결론

t-SNE도 두 클래스를 잘 분리하여 시각화(분포 형태 다름)

유방암 데이터셋처럼 지역 구조가 뚜렷한 데이터에서 효과적